In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

import sys, os
sys.path.insert(0, os.path.abspath(".."))
from python.encoder import Encoder

In [2]:
text = ""
with open('../dataset/1 - A Game of Thrones.txt', 'r', encoding='utf-8', errors='ignore') as f:
    text += f.read()

from python.tokenizer import Tokenizer
tokenizer = Tokenizer.load("../tokenizer.json")
tokens = tokenizer.encode(text)

In [3]:
class Head(nn.Module):

    def __init__(self, x_emb, head_emb, masking_enabled=False, cross_attention=False):
        super().__init__()
        self.key = nn.Linear(x_emb, head_emb)
        self.query = nn.Linear(x_emb, head_emb)
        self.value = nn.Linear(x_emb, head_emb)
        self.masking_enabled = masking_enabled
        self.cross_attention = cross_attention

    def forward(self, x, encoder_hidden=None):
        B, T, C = x.shape
        q = self.query(x)

        if self.cross_attention and encoder_hidden is not None:
            k = self.key(encoder_hidden)
            v = self.value(encoder_hidden)
        else:
            k = self.key(x)
            v = self.value(x)

        logits = q @ k.transpose(-2, -1)
        logits = logits / (k.shape[-1] ** 0.5)

        if self.masking_enabled:
            mask = torch.tril(torch.ones(T, T))
            logits = logits.masked_fill(mask == 0, float('-inf'))

        logits = F.softmax(logits, dim=-1)
        logits = logits @ v
        return logits

In [4]:
class MultiHeadAttention(nn.Module):

    def __init__(self, x_emb, head_emb, heads_num, cross_attention = False, masking_enabled = False):
        super().__init__()
        self.proj = nn.Linear(x_emb, x_emb)
        self.heads = nn.ModuleList([Head(x_emb, x_emb//heads_num, masking_enabled, cross_attention) for _ in range(heads_num)])

    def forward(self, x, encoder_logits = None):
        head_size = x.shape[-1] // len(self.heads)
        logits = torch.cat([
            head(x, encoder_logits)
            for i, head in enumerate(self.heads)
        ], dim=-1)
        logits = self.proj(logits)
        return logits

In [5]:
class MultiHeadBlock(nn.Module):

    def __init__(self, seq_len, x_emb, head_emb, heads_num, cross_attention = False, masking_enabled = False):
        super().__init__()
        self.heads = MultiHeadAttention(x_emb, head_emb, heads_num, cross_attention, masking_enabled)
        self.layer_norm = nn.LayerNorm((seq_len, x_emb))

    def forward(self, tokens, encoder_logits = None):
        logits = self.heads(tokens, encoder_logits)
        logits_norm = self.layer_norm(logits)
        return logits_norm + tokens

In [6]:
class FeedFwdBlock(nn.Module):

    def __init__(self, seq_len, x_emb):
        super().__init__()
        self.linear = nn.Linear(x_emb, x_emb)
        self.layer_norm = nn.LayerNorm((seq_len, x_emb))

    def forward(self, tokens):
        logits = self.linear(tokens)
        logits_norm = self.layer_norm(logits)
        return logits_norm + tokens

In [7]:
class DecoderArchitecture(nn.Module):

    def __init__(self, vocab_size, seq_len, x_emb, head_emb, heads_num, encoder_num):
        super().__init__()
        self.self_attention = MultiHeadBlock(seq_len, x_emb, head_emb, heads_num, cross_attention=False, masking_enabled=True)
        self.encoder = Encoder(vocab_size, x_emb, seq_len, heads_num, encoder_num)
        self.cross_attention = MultiHeadBlock(seq_len, x_emb, head_emb, heads_num, cross_attention=True, masking_enabled=True)
        self.feed_fwd = FeedFwdBlock(seq_len, x_emb)

    def forward(self, x, tokens):
        logits = self.self_attention(x)
        encoder_hidden = self.encoder.encode(tokens)
        logits = self.cross_attention(logits, encoder_hidden)
        logits = self.feed_fwd(logits)
        return logits

In [10]:
class Decoder(nn.Module):

    def __init__(self, vocab_size, seq_len, x_emb, head_emb, heads_num, decoder_num: int = 6, encoder_num: int = 6):
        super().__init__()
        self.seq_len = seq_len
        self.architecture = nn.ModuleList([DecoderArchitecture(vocab_size, seq_len, x_emb, head_emb, heads_num, encoder_num) for _ in range(decoder_num)])
        self.linear = nn.Linear(x_emb, vocab_size)
        self.look_up_table = nn.Parameter(torch.randn((vocab_size+2, x_emb)))
        self.postional_enc = nn.Parameter(torch.randn((seq_len, x_emb)))
        self.optimizer = torch.optim.AdamW(self.parameters(), lr=1e-3)

    def forward(self, tokens, target=None):
        loss = None
        B, T = tokens.shape
        x = self.look_up_table[tokens] + self.postional_enc[torch.arange(T)]

        for decoder in self.architecture:
            x = decoder(x, tokens)

        x = self.linear(x)

        if target is not None:
            target = torch.tensor(target)
            B, T, C = x.shape
            print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!  {x.view(B*T, C).shape}       {target.view(B*T).shape}")
            loss = F.cross_entropy(x.view(B*T, C), target.view(B*T))

        return x, loss

    def fit(self, tokens, epochs=100, batch_size=32):
        if len(tokens) % (self.seq_len+1) != 0:
            pad = self.seq_len + 1 - len(tokens) % (self.seq_len+1)
            tokens.extend([0] * pad)

        tokens = torch.tensor(tokens)
        tokens = tokens.view(-1, self.seq_len+1)
        x_chunks = tokens[:, :-1]
        y_chunks = tokens[:, 1:]

        for epoch in range(epochs):
            perm = torch.randperm(x_chunks.size(0))
            x = x_chunks[perm]
            y = y_chunks[perm]

            i = random.randint(0, len(x) - 1 - batch_size)
            x_batch = x[i: i+batch_size]
            y_batch = y[i: i+batch_size]

            output, loss = self.forward(x_batch, y_batch)
            loss.backward()
            print(f"epoch: {epoch}, loss: {loss.item():.4f}")
            self.optimizer.step()
            self.optimizer.zero_grad()

In [11]:
test_encode = Decoder(vocab_size=30000, seq_len=512, x_emb=32, head_emb=32, heads_num=4, decoder_num=6, encoder_num=6)
test_encode.fit(tokens=tokens, epochs=2, batch_size=32)

c:\Users\bhara\OneDrive\Desktop\transformer\python\encoder.py:92: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tokens = torch.tensor(tokens)
C:\Users\bhara\AppData\Local\Temp\ipykernel_47824\3758577084.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(target)


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!  torch.Size([16384, 30000])       torch.Size([16384])
epoch: 0, loss: 13.0110
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!  torch.Size([16384, 30000])       torch.Size([16384])
epoch: 1, loss: 12.9511
